This notebook tests `BVRA/beit_base_patch16_224.in1k_ft_fungitastic_224` on the image dataset

Please modify the `run` variable and the [configuration](#configuration)

# Setup

In [21]:
import os
import timm

from tqdm.notebook import tqdm

import torch
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
import torchvision.transforms as T
import torch.nn.functional as F

from sklearn.metrics import classification_report

import numpy as np
import pandas as pd

In [2]:
!pip install -r ../requirements.txt || pip install pytest pylint ipdb jupyterlab numpy pandas matplotlib seaborn scikit-learn tensorflow timm transformers keras_hub==0.26.0


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# How do you want to run it?
# run = "colab"
run = "local"

In [ ]:
if run == "colab":
    # Connect to google drive
    from google.colab import drive
    drive.mount('/content/drive')

    # Set the default root path of Kinoko project
    ROOT = "/content/drive/MyDrive/Colab Notebooks/lewagon/Kinoko"

    print(list_physical_devices('GPU'))
elif run == "local":
    ROOT = "../"
else:
    print("Error: variable `run` is set as an unknown type")

# Configuration

In [ ]:
# How many percentage of images of the image dataset you want to test on?
# IMAGE_PERCENT = 0.2 # 20%
IMAGE_PERCENT = 0.01

# Images

In [4]:
# Images transformations
train_transforms = T.Compose([T.Resize((224, 224)),
                              T.ToTensor(),
                              T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])


In [5]:
# Loading images
image_data_dir = f"{ROOT}/data/image_dataset"

dataset = ImageFolder(root=image_data_dir, 
                      transform=train_transforms)


In [11]:
# Keep 20% of images
total = len(dataset)
test_size = int(IMAGE_PERCENT * total)
train_size = total - test_size

_, test_dataset = random_split(dataset, [train_size, test_size], 
                                generator=torch.Generator().manual_seed(42))

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)


# Model

In [7]:
model = timm.create_model("hf-hub:BVRA/beit_base_patch16_224.in1k_ft_fungitastic_224", pretrained=True)

In [8]:
# Evaluate
model.eval()
correct = 0
total = 0
all_probs = []

In [24]:
# Get model's species class names
model_classes = model.pretrained_cfg.get('label_names')

samples = test_dataset.dataset.samples  # (path, class_idx) for all images
test_indices = test_dataset.indices     # indices into original dataset

results = []
batch_start = 0

model.eval()
with torch.no_grad():
    for images, _ in tqdm(test_loader, desc="Evaluating"):
        outputs = model(images)
        probs = F.softmax(outputs, dim=1)
        preds = probs.argmax(dim=1)

        for i in range(images.size(0)):
            path, _ = samples[test_indices[batch_start + i]]
            parts = path.split(os.sep)

            filename  = parts[-1]
            species   = parts[-2]
            edibility = parts[-3]  # "edible" or "poisonous"

            pred_idx = preds[i].item()
            results.append({
                "filename":          filename,
                "true_edibility":    edibility,
                "true_species":      species,
                "predicted_species": model_classes[pred_idx] if model_classes else pred_idx,
                "confidence":        probs[i, pred_idx].item(),
            })

        batch_start += images.size(0)

all_preds = pd.DataFrame(results)

Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

In [25]:
all_preds

,filename,true_edibility,true_species,predicted_species,confidence
0,Galerina_sulciceps1.png,poisonous,Galerina_sulciceps,1528,0.271664
1,Cortinarius_caperatus33.png,edible,Cortinarius_caperatus,539,0.998521
2,Lepiota_brunneoincarnata27.png,poisonous,Lepiota_brunneoincarnata,763,0.565455
3,Galerina_marginata4.png,poisonous,Galerina_marginata,2718,0.498537
4,Inocybe_sambucina3.png,poisonous,Inocybe_sambucina,1279,0.246723
...,...,...,...,...,...
74,Hebeloma_sinapizans3.png,poisonous,Hebeloma_sinapizans,2185,0.271242
75,Amanita_exitialis22.png,poisonous,Amanita_exitialis,1584,0.930382
76,Paralepistopsis_amoenolens27.png,poisonous,Paralepistopsis_amoenolens,1387,0.157529
77,Galerina_sulciceps21.png,poisonous,Galerina_sulciceps,941,0.882720


In [ ]:
# TODO: Link everything with table dataset and get predicted e/p label and know the accuracy

              precision    recall  f1-score   support

      edible       0.00      0.00      0.00      21.0
   poisonous       0.00      0.00      0.00      58.0

   micro avg       0.00      0.00      0.00      79.0
   macro avg       0.00      0.00      0.00      79.0
weighted avg       0.00      0.00      0.00      79.0



/home/clement/.pyenv/versions/kinoko/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/clement/.pyenv/versions/kinoko/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/clement/.pyenv/versions/kinoko/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.cap